In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

# Feature Engineering

This notebook creates financial features from the prepared Income Statement and Cash Flow datasets.

## Outputs
- `income_statement_features.csv`
- `cashflow_features.csv`

The feature engineering process is designed to avoid future information leakage. Features for each company-year observation are calculated only from information available at or before the relevant Publish Date.

In [6]:
output_path = "/content/drive/MyDrive/Data Handling and Machine Learning/week_03/outputs"

financial_path = output_path + "/financial_features"

print("Financial data path:")
print(financial_path)

Financial data path:
/content/drive/MyDrive/Data Handling and Machine Learning/week_03/outputs/financial_features


In [7]:
income_path = financial_path + "/income_statement.csv"
cashflow_path = financial_path + "/cash_flow.csv"

income_df = pd.read_csv(income_path)
cashflow_df = pd.read_csv(cashflow_path)

print("Income Statement shape:", income_df.shape)
print("Cash Flow shape:", cashflow_df.shape)

Income Statement shape: (500, 10)
Cash Flow shape: (500, 9)


In [8]:
print("Income Statement columns:")
print(income_df.columns.tolist())

print("\nCash Flow columns:")
print(cashflow_df.columns.tolist())

Income Statement columns:
['Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income']

Cash Flow columns:
['Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Operating Cash Flow', 'Capital Expenditure', 'Free Cash Flow']


In [9]:
print("=== INCOME STATEMENT INFO ===")
income_df.info()

print("\n=== INCOME STATEMENT MISSING VALUES ===")
print(income_df.isnull().sum())

print("\n=== SAMPLE ROWS ===")
display(income_df.head())

=== INCOME STATEMENT INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Ticker            500 non-null    object 
 1   SimFinId          500 non-null    int64  
 2   Currency          500 non-null    object 
 3   Fiscal Year       500 non-null    int64  
 4   Fiscal Period     500 non-null    object 
 5   Report Date       500 non-null    object 
 6   Revenue           500 non-null    float64
 7   Gross Profit      464 non-null    float64
 8   Operating Income  500 non-null    float64
 9   Net Income        500 non-null    int64  
dtypes: float64(3), int64(3), object(4)
memory usage: 39.2+ KB

=== INCOME STATEMENT MISSING VALUES ===
Ticker               0
SimFinId             0
Currency             0
Fiscal Year          0
Fiscal Period        0
Report Date          0
Revenue              0
Gross Profit        36
Operating Inc

,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Revenue,Gross Profit,Operating Income,Net Income
0,ABBV,61199,USD,2020,FY,2020-12-31,4.580400e+10,3.041700e+10,1.273900e+10,4616000000
1,ABBV,61199,USD,2021,FY,2021-12-31,5.619700e+10,3.875100e+10,1.904800e+10,11542000000
2,ABBV,61199,USD,2022,FY,2022-12-31,5.805400e+10,4.064000e+10,1.881400e+10,11836000000
3,ABBV,61199,USD,2023,FY,2023-12-31,5.431800e+10,3.390300e+10,1.353500e+10,4863000000
4,ABBV,61199,USD,2024,FY,2024-12-31,5.633400e+10,3.943000e+10,1.189400e+10,4278000000


In [10]:
raw_path = "/content/drive/MyDrive/Data Handling and Machine Learning/week_03/raw"

print("Files in raw folder:\n")

for file in os.listdir(raw_path):
    print(file)

Files in raw folder:

share_prices.csv
income_statement.csv
cash_flow.csv
company_info.csv
balance_sheet.csv
100_companies.xlsx


In [11]:
raw_income_path = raw_path + "/income_statement.csv"

raw_income_df = pd.read_csv(raw_income_path, sep=";")

print("Raw Income Statement shape:", raw_income_df.shape)

print("\nRaw Income Statement columns:")
print(raw_income_df.columns.tolist())

Raw Income Statement shape: (16844, 28)

Raw Income Statement columns:
['Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Expenses', 'Selling, General & Administrative', 'Research & Development', 'Depreciation & Amortization', 'Operating Income (Loss)', 'Non-Operating Income (Loss)', 'Interest Expense, Net', 'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)', 'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net', 'Income (Loss) from Continuing Operations', 'Net Extraordinary Gains (Losses)', 'Net Income', 'Net Income (Common)']


In [ ]:
# Select only the columns needed to recover Publish Date
publish_dates = raw_income_df[
    ['SimFinId', 'Fiscal Year', 'Publish Date']
].copy()

# Check for duplicate company-year records
print(
    "Duplicate SimFinId-Fiscal Year records:",
    publish_dates.duplicated(
        subset=['SimFinId', 'Fiscal Year']
    ).sum()
)

# Preview
display(publish_dates.head())

Duplicate SimFinId-Fiscal Year records: 0


,SimFinId,Fiscal Year,Publish Date
0,45846,2020,2020-12-18
1,45846,2021,2021-12-17
2,45846,2022,2022-12-21
3,45846,2023,2023-12-20
4,45846,2024,2024-12-20


In [ ]:
income_df = income_df.merge(
    publish_dates,
    on=['SimFinId', 'Fiscal Year'],
    how='left'
)

print("Income Statement shape after merge:", income_df.shape)

print("\nMissing Publish Date values:")
print(income_df['Publish Date'].isnull().sum())

display(
    income_df[
        ['Ticker', 'SimFinId', 'Fiscal Year', 'Report Date', 'Publish Date']
    ].head()
)

Income Statement shape after merge: (500, 11)

Missing Publish Date values:
0


,Ticker,SimFinId,Fiscal Year,Report Date,Publish Date
0,ABBV,61199,2020,2020-12-31,2021-02-19
1,ABBV,61199,2021,2021-12-31,2022-02-18
2,ABBV,61199,2022,2022-12-31,2023-02-17
3,ABBV,61199,2023,2023-12-31,2024-02-20
4,ABBV,61199,2024,2024-12-31,2025-02-14


## Income Statement Feature Engineering

The prepared Income Statement dataset is used to create financial features for each company and fiscal year.

The following features will be calculated:

- **Revenue Growth:** Year-over-year change in revenue.
- **Gross Profit Margin:** Gross Profit divided by Revenue.
- **Operating Profit Margin:** Operating Income divided by Revenue.
- **Net Profit Margin:** Net Income divided by Revenue.
- **Operating Income Growth:** Year-over-year change in Operating Income.
- **Net Income Growth:** Year-over-year change in Net Income.

To prevent future information leakage, the data is sorted by `SimFinId` and `Fiscal Year` before calculating year-over-year growth. Growth calculations use only the previous fiscal year's financial values for the same company.

Division-by-zero and missing values will also be checked before the final feature dataset is saved.

### Sort Data for Year-over-Year Calculations

The observations are sorted by company (`SimFinId`) and `Fiscal Year` so that previous-year values can be correctly identified when calculating growth features.

In [ ]:
income_df = income_df.sort_values(
    by=['SimFinId', 'Fiscal Year']
).reset_index(drop=True)

display(
    income_df[
        ['Ticker', 'SimFinId', 'Fiscal Year', 'Publish Date',
         'Revenue', 'Operating Income', 'Net Income']
    ].head(10)
)

,Ticker,SimFinId,Fiscal Year,Publish Date,Revenue,Operating Income,Net Income
0,GOOG,18,2020,2021-02-03,1.825270e+11,4.122400e+10,40269000000
1,GOOG,18,2021,2022-02-02,2.576370e+11,7.871400e+10,76033000000
2,GOOG,18,2022,2023-02-03,2.828360e+11,7.484200e+10,59972000000
3,GOOG,18,2023,2024-01-31,3.073940e+11,8.429300e+10,73795000000
4,GOOG,18,2024,2025-02-05,3.500180e+11,1.123900e+11,100118000000
5,ADBE,14099,2020,2021-01-15,1.286800e+10,4.237000e+09,5260000000
6,ADBE,14099,2021,2022-01-21,1.578500e+10,5.802000e+09,4822000000
7,ADBE,14099,2022,2023-01-17,1.760600e+10,6.098000e+09,4756000000
8,ADBE,14099,2023,2024-01-17,1.940900e+10,6.650000e+09,5428000000
9,ADBE,14099,2024,2025-01-13,2.150500e+10,7.741000e+09,5560000000


### Revenue Growth

Revenue Growth measures the year-over-year percentage change in a company's revenue.

It is calculated as:

**Revenue Growth = (Current Year Revenue - Previous Year Revenue) / Previous Year Revenue**

The calculation is performed separately for each company using `SimFinId`. The first fiscal year for each company will have a missing Revenue Growth value because no previous-year observation is available.

In [ ]:
# Get previous year's revenue for each company
income_df['Previous Revenue'] = (
    income_df.groupby('SimFinId')['Revenue'].shift(1)
)

# Calculate year-over-year Revenue Growth
income_df['Revenue Growth'] = np.where(
    income_df['Previous Revenue'] != 0,
    (income_df['Revenue'] - income_df['Previous Revenue'])
    / income_df['Previous Revenue'],
    np.nan
)

# Display sample results
display(
    income_df[
        ['Ticker', 'Fiscal Year', 'Revenue',
         'Previous Revenue', 'Revenue Growth']
    ].head(10)
)

,Ticker,Fiscal Year,Revenue,Previous Revenue,Revenue Growth
0,GOOG,2020,1.825270e+11,NaN,NaN
1,GOOG,2021,2.576370e+11,1.825270e+11,0.411501
2,GOOG,2022,2.828360e+11,2.576370e+11,0.097808
3,GOOG,2023,3.073940e+11,2.828360e+11,0.086828
4,GOOG,2024,3.500180e+11,3.073940e+11,0.138662
5,ADBE,2020,1.286800e+10,NaN,NaN
6,ADBE,2021,1.578500e+10,1.286800e+10,0.226686
7,ADBE,2022,1.760600e+10,1.578500e+10,0.115363
8,ADBE,2023,1.940900e+10,1.760600e+10,0.102408
9,ADBE,2024,2.150500e+10,1.940900e+10,0.107991


### Profit Margin Features

The following profitability ratios are calculated using Revenue as the denominator:

- **Gross Profit Margin = Gross Profit / Revenue**
- **Operating Profit Margin = Operating Income / Revenue**
- **Net Profit Margin = Net Income / Revenue**

If Revenue is zero, the result is set to `NaN` to avoid division-by-zero errors.

In [ ]:
# Calculate profit margin features safely
income_df['Gross Profit Margin'] = np.where(
    income_df['Revenue'] != 0,
    income_df['Gross Profit'] / income_df['Revenue'],
    np.nan
)

income_df['Operating Profit Margin'] = np.where(
    income_df['Revenue'] != 0,
    income_df['Operating Income'] / income_df['Revenue'],
    np.nan
)

income_df['Net Profit Margin'] = np.where(
    income_df['Revenue'] != 0,
    income_df['Net Income'] / income_df['Revenue'],
    np.nan
)

# Display sample results
display(
    income_df[
        [
            'Ticker',
            'Fiscal Year',
            'Revenue',
            'Gross Profit Margin',
            'Operating Profit Margin',
            'Net Profit Margin'
        ]
    ].head(10)
)

,Ticker,Fiscal Year,Revenue,Gross Profit Margin,Operating Profit Margin,Net Profit Margin
0,GOOG,2020,1.825270e+11,0.535784,0.225852,0.220619
1,GOOG,2021,2.576370e+11,0.569398,0.305523,0.295117
2,GOOG,2022,2.828360e+11,0.553794,0.264613,0.212038
3,GOOG,2023,3.073940e+11,0.566250,0.274218,0.240066
4,GOOG,2024,3.500180e+11,0.582004,0.321098,0.286037
5,ADBE,2020,1.286800e+10,0.866180,0.329266,0.408766
6,ADBE,2021,1.578500e+10,0.881850,0.367564,0.305480
7,ADBE,2022,1.760600e+10,0.877031,0.346359,0.270135
8,ADBE,2023,1.940900e+10,0.878716,0.342625,0.279664
9,ADBE,2024,2.150500e+10,0.890351,0.359963,0.258545


### Operating Income and Net Income Growth

Operating Income Growth and Net Income Growth measure the year-over-year change in operating income and net income.

They are calculated as:

**Operating Income Growth = (Current Operating Income - Previous Operating Income) / Previous Operating Income**

**Net Income Growth = (Current Net Income - Previous Net Income) / Previous Net Income**

The calculations are performed separately for each company using `SimFinId`. The first available fiscal year for each company will have a missing growth value because there is no previous-year observation.

If a previous-year value is zero, the corresponding growth value is set to `NaN` to avoid division by zero.

In [ ]:
# Get previous year's values for each company
income_df['Previous Operating Income'] = (
    income_df.groupby('SimFinId')['Operating Income'].shift(1)
)

income_df['Previous Net Income'] = (
    income_df.groupby('SimFinId')['Net Income'].shift(1)
)

# Calculate Operating Income Growth
income_df['Operating Income Growth'] = np.where(
    income_df['Previous Operating Income'] != 0,
    (income_df['Operating Income'] - income_df['Previous Operating Income'])
    / income_df['Previous Operating Income'],
    np.nan
)

# Calculate Net Income Growth
income_df['Net Income Growth'] = np.where(
    income_df['Previous Net Income'] != 0,
    (income_df['Net Income'] - income_df['Previous Net Income'])
    / income_df['Previous Net Income'],
    np.nan
)

# Display sample results
display(
    income_df[
        [
            'Ticker',
            'Fiscal Year',
            'Operating Income',
            'Previous Operating Income',
            'Operating Income Growth',
            'Net Income',
            'Previous Net Income',
            'Net Income Growth'
        ]
    ].head(10)
)

,Ticker,Fiscal Year,Operating Income,Previous Operating Income,Operating Income Growth,Net Income,Previous Net Income,Net Income Growth
0,GOOG,2020,4.122400e+10,NaN,NaN,40269000000,NaN,NaN
1,GOOG,2021,7.871400e+10,4.122400e+10,0.909422,76033000000,4.026900e+10,0.888127
2,GOOG,2022,7.484200e+10,7.871400e+10,-0.049191,59972000000,7.603300e+10,-0.211237
3,GOOG,2023,8.429300e+10,7.484200e+10,0.126279,73795000000,5.997200e+10,0.230491
4,GOOG,2024,1.123900e+11,8.429300e+10,0.333325,100118000000,7.379500e+10,0.356704
5,ADBE,2020,4.237000e+09,NaN,NaN,5260000000,NaN,NaN
6,ADBE,2021,5.802000e+09,4.237000e+09,0.369365,4822000000,5.260000e+09,-0.083270
7,ADBE,2022,6.098000e+09,5.802000e+09,0.051017,4756000000,4.822000e+09,-0.013687
8,ADBE,2023,6.650000e+09,6.098000e+09,0.090521,5428000000,4.756000e+09,0.141295
9,ADBE,2024,7.741000e+09,6.650000e+09,0.164060,5560000000,5.428000e+09,0.024318


### Income Statement Feature Validation

Before creating the final feature dataset, the engineered features are checked for missing values, infinite values, and unexpected results.

Missing values in the first year of each company's growth features are expected because no previous-year observation is available. Missing Gross Profit Margin values may also occur where Gross Profit was missing in the prepared dataset.

Infinite values are checked to ensure that division-by-zero has not produced invalid feature values.

In [ ]:
income_features = [
    'Revenue Growth',
    'Gross Profit Margin',
    'Operating Profit Margin',
    'Net Profit Margin',
    'Operating Income Growth',
    'Net Income Growth'
]

print("=== Missing Values ===")
print(income_df[income_features].isnull().sum())

print("\n=== Infinite Values ===")
print(
    np.isinf(
        income_df[income_features].select_dtypes(include=np.number)
    ).sum()
)

print("\n=== Feature Summary ===")
display(income_df[income_features].describe())

=== Missing Values ===
Revenue Growth             100
Gross Profit Margin         36
Operating Profit Margin      0
Net Profit Margin            0
Operating Income Growth    100
Net Income Growth          100
dtype: int64

=== Infinite Values ===
Revenue Growth             0
Gross Profit Margin        0
Operating Profit Margin    0
Net Profit Margin          0
Operating Income Growth    0
Net Income Growth          0
dtype: int64

=== Feature Summary ===


,Revenue Growth,Gross Profit Margin,Operating Profit Margin,Net Profit Margin,Operating Income Growth,Net Income Growth
count,400.000000,464.000000,500.000000,500.000000,400.000000,400.000000
mean,0.190069,0.499146,0.173539,0.141253,0.300753,0.396697
std,1.131576,0.236792,0.265130,0.352462,15.942089,4.455517
min,-0.644500,-0.388463,-1.969874,-3.211650,-152.870968,-17.334672
25%,0.014177,0.326187,0.091605,0.065425,-0.121561,-0.292018
50%,0.081773,0.543336,0.186387,0.130647,0.082418,0.044273
75%,0.172550,0.692695,0.302792,0.234771,0.257466,0.368392
max,22.002491,0.990037,0.861913,3.364554,274.813736,49.609756


### Final Income Statement Feature Dataset

The required identifiers, engineered features, and original financial values are retained in the final Income Statement feature dataset.

The original financial values are included to support verification of the engineered features. No rows are removed because of expected missing feature values.

In [ ]:
income_statement_features = income_df[
    [
        'SimFinId',
        'Ticker',
        'Fiscal Year',
        'Publish Date',
        'Revenue',
        'Gross Profit',
        'Operating Income',
        'Net Income',
        'Revenue Growth',
        'Gross Profit Margin',
        'Operating Profit Margin',
        'Net Profit Margin',
        'Operating Income Growth',
        'Net Income Growth'
    ]
].copy()

print("Final Income Statement feature shape:",
      income_statement_features.shape)

display(income_statement_features.head())

Final Income Statement feature shape: (500, 14)


,SimFinId,Ticker,Fiscal Year,Publish Date,Revenue,Gross Profit,Operating Income,Net Income,Revenue Growth,Gross Profit Margin,Operating Profit Margin,Net Profit Margin,Operating Income Growth,Net Income Growth
0,18,GOOG,2020,2021-02-03,1.825270e+11,9.779500e+10,4.122400e+10,40269000000,NaN,0.535784,0.225852,0.220619,NaN,NaN
1,18,GOOG,2021,2022-02-02,2.576370e+11,1.466980e+11,7.871400e+10,76033000000,0.411501,0.569398,0.305523,0.295117,0.909422,0.888127
2,18,GOOG,2022,2023-02-03,2.828360e+11,1.566330e+11,7.484200e+10,59972000000,0.097808,0.553794,0.264613,0.212038,-0.049191,-0.211237
3,18,GOOG,2023,2024-01-31,3.073940e+11,1.740620e+11,8.429300e+10,73795000000,0.086828,0.566250,0.274218,0.240066,0.126279,0.230491
4,18,GOOG,2024,2025-02-05,3.500180e+11,2.037120e+11,1.123900e+11,100118000000,0.138662,0.582004,0.321098,0.286037,0.333325,0.356704


### Save Income Statement Features

A final structural check is performed before saving the dataset. The completed Income Statement feature dataset is then saved as a new CSV file without modifying the original prepared financial data.

In [ ]:
# Final checks
print("Number of rows:", len(income_statement_features))
print("Unique companies:", income_statement_features['SimFinId'].nunique())

print(
    "Duplicate company-year records:",
    income_statement_features.duplicated(
        subset=['SimFinId', 'Fiscal Year']
    ).sum()
)

print(
    "Missing Publish Dates:",
    income_statement_features['Publish Date'].isnull().sum()
)

Number of rows: 500
Unique companies: 100
Duplicate company-year records: 0
Missing Publish Dates: 0


In [5]:
income_features_path = financial_path + "/income_statement_features.csv"

income_statement_features.to_csv(
    income_features_path,
    index=False
)

print("Saved successfully:")
print(income_features_path)

NameError: name 'financial_path' is not defined

## Cash Flow Feature Engineering

The prepared Cash Flow dataset is used to create cash-flow-related financial features for each company and fiscal year.

The following features will be calculated:

- **Free Cash Flow:** Operating Cash Flow minus Capital Expenditure.
- **Operating Cash Flow Growth:** Year-over-year change in Operating Cash Flow.
- **Free Cash Flow Growth:** Year-over-year change in Free Cash Flow.
- **FCF Margin:** Free Cash Flow divided by Revenue.

The calculations will be performed separately for each company using `SimFinId`. Publish Date and Revenue will be obtained using the appropriate company-year identifiers where required.

Division-by-zero, missing values, duplicate records, and infinite values will be checked before the final dataset is saved.

In [4]:
cashflow_path = financial_path + "/cash_flow.csv"

cashflow_df = pd.read_csv(cashflow_path)

print("Cash Flow dataset loaded successfully.")
print("Shape:", cashflow_df.shape)

NameError: name 'financial_path' is not defined

In [ ]:
print("=== CASH FLOW INFO ===")
cashflow_df.info()

print("\n=== CASH FLOW MISSING VALUES ===")
print(cashflow_df.isnull().sum())

print("\n=== SAMPLE ROWS ===")
display(cashflow_df.head())

=== CASH FLOW INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Ticker               500 non-null    object 
 1   SimFinId             500 non-null    int64  
 2   Currency             500 non-null    object 
 3   Fiscal Year          500 non-null    int64  
 4   Fiscal Period        500 non-null    object 
 5   Report Date          500 non-null    object 
 6   Operating Cash Flow  500 non-null    float64
 7   Capital Expenditure  482 non-null    float64
 8   Free Cash Flow       482 non-null    float64
dtypes: float64(3), int64(2), object(4)
memory usage: 35.3+ KB

=== CASH FLOW MISSING VALUES ===
Ticker                  0
SimFinId                0
Currency                0
Fiscal Year             0
Fiscal Period           0
Report Date             0
Operating Cash Flow     0
Capital Expenditure    18
Free Cash Flow    

,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Operating Cash Flow,Capital Expenditure,Free Cash Flow
0,ABBV,61199,USD,2020,FY,2020-12-31,1.758800e+10,-798000000.0,1.679000e+10
1,ABBV,61199,USD,2021,FY,2021-12-31,2.277700e+10,-787000000.0,2.199000e+10
2,ABBV,61199,USD,2022,FY,2022-12-31,2.494300e+10,-695000000.0,2.424800e+10
3,ABBV,61199,USD,2023,FY,2023-12-31,2.283900e+10,-777000000.0,2.206200e+10
4,ABBV,61199,USD,2024,FY,2024-12-31,1.880600e+10,-974000000.0,1.783200e+10


### Add Publish Date and Revenue

The prepared Cash Flow dataset does not contain `Publish Date` or `Revenue`.

`Publish Date` is required as an identifier in the final Cash Flow feature dataset, while `Revenue` is required to calculate FCF Margin.

These values are obtained from the Income Statement using `SimFinId` and `Fiscal Year` as the company-year matching keys.

In [ ]:
income_features_path = financial_path + "/income_statement_features.csv"

income_lookup = pd.read_csv(
    income_features_path,
    usecols=['SimFinId', 'Fiscal Year', 'Publish Date', 'Revenue']
)

print("Lookup shape:", income_lookup.shape)

print(
    "Duplicate company-year records:",
    income_lookup.duplicated(
        subset=['SimFinId', 'Fiscal Year']
    ).sum()
)

print("\nMissing values:")
print(income_lookup.isnull().sum())

display(income_lookup.head())

Lookup shape: (500, 4)
Duplicate company-year records: 0

Missing values:
SimFinId        0
Fiscal Year     0
Publish Date    0
Revenue         0
dtype: int64


,SimFinId,Fiscal Year,Publish Date,Revenue
0,18,2020,2021-02-03,1.825270e+11
1,18,2021,2022-02-02,2.576370e+11
2,18,2022,2023-02-03,2.828360e+11
3,18,2023,2024-01-31,3.073940e+11
4,18,2024,2025-02-05,3.500180e+11


### Merge Publish Date and Revenue

`Publish Date` and `Revenue` are merged into the Cash Flow dataset using `SimFinId` and `Fiscal Year`.

This preserves the original 500 company-year observations while adding the fields required for the final output and FCF Margin calculation.

In [ ]:
cashflow_df = cashflow_df.merge(
    income_lookup,
    on=['SimFinId', 'Fiscal Year'],
    how='left'
)

print("Cash Flow shape after merge:", cashflow_df.shape)

print("\nMissing Publish Date values:")
print(cashflow_df['Publish Date'].isnull().sum())

print("\nMissing Revenue values:")
print(cashflow_df['Revenue'].isnull().sum())

display(
    cashflow_df[
        ['Ticker', 'SimFinId', 'Fiscal Year',
         'Publish Date', 'Operating Cash Flow',
         'Capital Expenditure', 'Revenue']
    ].head()
)

Cash Flow shape after merge: (500, 11)

Missing Publish Date values:
0

Missing Revenue values:
0


,Ticker,SimFinId,Fiscal Year,Publish Date,Operating Cash Flow,Capital Expenditure,Revenue
0,ABBV,61199,2020,2021-02-19,1.758800e+10,-798000000.0,4.580400e+10
1,ABBV,61199,2021,2022-02-18,2.277700e+10,-787000000.0,5.619700e+10
2,ABBV,61199,2022,2023-02-17,2.494300e+10,-695000000.0,5.805400e+10
3,ABBV,61199,2023,2024-02-20,2.283900e+10,-777000000.0,5.431800e+10
4,ABBV,61199,2024,2025-02-14,1.880600e+10,-974000000.0,5.633400e+10


### Free Cash Flow

Free Cash Flow (FCF) represents the cash remaining after capital expenditure.

The feature requirement defines:

**Free Cash Flow = Operating Cash Flow - Capital Expenditure**

However, Capital Expenditure in the prepared SimFin dataset is recorded as a negative cash outflow. Therefore, directly subtracting the negative value would incorrectly increase Free Cash Flow.

To account for the dataset's sign convention, Free Cash Flow is calculated as:

**Free Cash Flow = Operating Cash Flow + Capital Expenditure**

This calculation is also consistent with the Free Cash Flow values already present in the prepared dataset. Missing Capital Expenditure values are retained as missing rather than removing the affected observations.

In [ ]:
# Recalculate FCF using the dataset's negative CapEx convention
cashflow_df['Calculated Free Cash Flow'] = (
    cashflow_df['Operating Cash Flow']
    + cashflow_df['Capital Expenditure']
)

# Compare calculated FCF with the existing prepared FCF
fcf_difference = (
    cashflow_df['Calculated Free Cash Flow']
    - cashflow_df['Free Cash Flow']
)

print(
    "Rows where calculated and existing FCF differ:",
    (fcf_difference.abs() > 0.01).sum()
)

print(
    "Missing calculated FCF values:",
    cashflow_df['Calculated Free Cash Flow'].isnull().sum()
)

display(
    cashflow_df[
        [
            'Ticker',
            'Fiscal Year',
            'Operating Cash Flow',
            'Capital Expenditure',
            'Free Cash Flow',
            'Calculated Free Cash Flow'
        ]
    ].head(10)
)

Rows where calculated and existing FCF differ: 0
Missing calculated FCF values: 18


,Ticker,Fiscal Year,Operating Cash Flow,Capital Expenditure,Free Cash Flow,Calculated Free Cash Flow
0,ABBV,2020,1.758800e+10,-798000000.0,1.679000e+10,1.679000e+10
1,ABBV,2021,2.277700e+10,-787000000.0,2.199000e+10,2.199000e+10
2,ABBV,2022,2.494300e+10,-695000000.0,2.424800e+10,2.424800e+10
3,ABBV,2023,2.283900e+10,-777000000.0,2.206200e+10,2.206200e+10
4,ABBV,2024,1.880600e+10,-974000000.0,1.783200e+10,1.783200e+10
5,ABNB,2020,-7.400000e+08,-37000000.0,-7.770000e+08,-7.770000e+08
6,ABNB,2021,2.313000e+09,NaN,NaN,NaN
7,ABNB,2022,3.430000e+09,NaN,NaN,NaN
8,ABNB,2023,3.884000e+09,NaN,NaN,NaN
9,ABNB,2024,4.518000e+09,NaN,NaN,NaN


### Cash Flow Growth Features

The Cash Flow observations are sorted by `SimFinId` and `Fiscal Year` before calculating year-over-year growth.

Two growth features are calculated:

- **Operating Cash Flow Growth = (Current Operating Cash Flow - Previous Operating Cash Flow) / Previous Operating Cash Flow**
- **Free Cash Flow Growth = (Current Free Cash Flow - Previous Free Cash Flow) / Previous Free Cash Flow**

The calculations are performed separately for each company. The first available year for each company will have a missing growth value because no previous-year observation is available.

If the previous-year value is zero, the corresponding growth value is set to `NaN` to avoid division by zero.

In [ ]:
# Sort by company and fiscal year
cashflow_df = cashflow_df.sort_values(
    by=['SimFinId', 'Fiscal Year']
).reset_index(drop=True)

# Use the verified calculated FCF
cashflow_df['Free Cash Flow'] = cashflow_df['Calculated Free Cash Flow']

# Get previous-year values
cashflow_df['Previous Operating Cash Flow'] = (
    cashflow_df.groupby('SimFinId')['Operating Cash Flow'].shift(1)
)

cashflow_df['Previous Free Cash Flow'] = (
    cashflow_df.groupby('SimFinId')['Free Cash Flow'].shift(1)
)

# Calculate Operating Cash Flow Growth
cashflow_df['Operating Cash Flow Growth'] = np.where(
    cashflow_df['Previous Operating Cash Flow'] != 0,
    (
        cashflow_df['Operating Cash Flow']
        - cashflow_df['Previous Operating Cash Flow']
    ) / cashflow_df['Previous Operating Cash Flow'],
    np.nan
)

# Calculate Free Cash Flow Growth
cashflow_df['Free Cash Flow Growth'] = np.where(
    cashflow_df['Previous Free Cash Flow'] != 0,
    (
        cashflow_df['Free Cash Flow']
        - cashflow_df['Previous Free Cash Flow']
    ) / cashflow_df['Previous Free Cash Flow'],
    np.nan
)

display(
    cashflow_df[
        [
            'Ticker',
            'Fiscal Year',
            'Operating Cash Flow',
            'Previous Operating Cash Flow',
            'Operating Cash Flow Growth',
            'Free Cash Flow',
            'Previous Free Cash Flow',
            'Free Cash Flow Growth'
        ]
    ].head(10)
)

,Ticker,Fiscal Year,Operating Cash Flow,Previous Operating Cash Flow,Operating Cash Flow Growth,Free Cash Flow,Previous Free Cash Flow,Free Cash Flow Growth
0,GOOG,2020,6.512400e+10,NaN,NaN,4.284300e+10,NaN,NaN
1,GOOG,2021,9.165200e+10,6.512400e+10,0.407346,6.701200e+10,4.284300e+10,0.564129
2,GOOG,2022,9.149500e+10,9.165200e+10,-0.001713,6.001000e+10,6.701200e+10,-0.104489
3,GOOG,2023,1.017460e+11,9.149500e+10,0.112039,6.949500e+10,6.001000e+10,0.158057
4,GOOG,2024,1.252990e+11,1.017460e+11,0.231488,7.276400e+10,6.949500e+10,0.047039
5,ADBE,2020,5.727000e+09,NaN,NaN,5.308000e+09,NaN,NaN
6,ADBE,2021,7.230000e+09,5.727000e+09,0.262441,6.882000e+09,5.308000e+09,0.296534
7,ADBE,2022,7.838000e+09,7.230000e+09,0.084094,7.396000e+09,6.882000e+09,0.074688
8,ADBE,2023,7.302000e+09,7.838000e+09,-0.068385,6.942000e+09,7.396000e+09,-0.061385
9,ADBE,2024,8.056000e+09,7.302000e+09,0.103259,7.873000e+09,6.942000e+09,0.134111


### Free Cash Flow Margin

FCF Margin measures the amount of Free Cash Flow generated relative to Revenue.

It is calculated as:

**FCF Margin = Free Cash Flow / Revenue**

Revenue was obtained from the corresponding Income Statement company-year observation. If Revenue is zero, FCF Margin is set to `NaN` to avoid division by zero. Missing Free Cash Flow values are retained as missing.

In [ ]:
# Calculate FCF Margin safely
cashflow_df['FCF Margin'] = np.where(
    cashflow_df['Revenue'] != 0,
    cashflow_df['Free Cash Flow'] / cashflow_df['Revenue'],
    np.nan
)

# Display sample results
display(
    cashflow_df[
        [
            'Ticker',
            'Fiscal Year',
            'Free Cash Flow',
            'Revenue',
            'FCF Margin'
        ]
    ].head(10)
)

,Ticker,Fiscal Year,Free Cash Flow,Revenue,FCF Margin
0,GOOG,2020,4.284300e+10,1.825270e+11,0.234721
1,GOOG,2021,6.701200e+10,2.576370e+11,0.260102
2,GOOG,2022,6.001000e+10,2.828360e+11,0.212172
3,GOOG,2023,6.949500e+10,3.073940e+11,0.226078
4,GOOG,2024,7.276400e+10,3.500180e+11,0.207886
5,ADBE,2020,5.308000e+09,1.286800e+10,0.412496
6,ADBE,2021,6.882000e+09,1.578500e+10,0.435984
7,ADBE,2022,7.396000e+09,1.760600e+10,0.420084
8,ADBE,2023,6.942000e+09,1.940900e+10,0.357669
9,ADBE,2024,7.873000e+09,2.150500e+10,0.366101


### Cash Flow Feature Validation

Before creating the final Cash Flow feature dataset, the engineered features are checked for missing values, infinite values, and unexpected results.

Missing values in growth features may occur for the first available year of each company because no previous-year observation exists. Additional missing Free Cash Flow-related features may occur where Capital Expenditure was missing in the prepared dataset.

Infinite values are checked to ensure that division-by-zero has not produced invalid feature values.

In [ ]:
cashflow_features_list = [
    'Free Cash Flow',
    'Operating Cash Flow Growth',
    'Free Cash Flow Growth',
    'FCF Margin'
]

print("=== Missing Values ===")
print(cashflow_df[cashflow_features_list].isnull().sum())

print("\n=== Infinite Values ===")
print(
    np.isinf(
        cashflow_df[cashflow_features_list].select_dtypes(include=np.number)
    ).sum()
)

print("\n=== Feature Summary ===")
display(cashflow_df[cashflow_features_list].describe())

=== Missing Values ===
Free Cash Flow                 18
Operating Cash Flow Growth    100
Free Cash Flow Growth         118
FCF Margin                     18
dtype: int64

=== Infinite Values ===
Free Cash Flow                0
Operating Cash Flow Growth    0
Free Cash Flow Growth         0
FCF Margin                    0
dtype: int64

=== Feature Summary ===


,Free Cash Flow,Operating Cash Flow Growth,Free Cash Flow Growth,FCF Margin
count,4.820000e+02,400.000000,382.000000,482.000000
mean,7.792141e+09,1.282418,1.692771,0.152838
std,1.200007e+10,21.670553,32.229951,0.331393
min,-1.683400e+10,-15.636364,-18.600000,-4.168946
25%,1.433000e+09,-0.125352,-0.227608,0.072135
50%,4.682500e+09,0.050366,0.032541,0.150326
75%,9.461750e+09,0.296574,0.307909,0.271140
max,7.407100e+10,432.430435,627.500000,2.439601


### Final Cash Flow Feature Dataset

The final Cash Flow feature dataset contains the required company-year identifiers, engineered Cash Flow features, and original financial values needed for verification.

Missing values caused by unavailable Capital Expenditure or unavailable previous-year observations are retained rather than removing the affected rows.

In [ ]:
cashflow_features = cashflow_df[
    [
        'SimFinId',
        'Ticker',
        'Fiscal Year',
        'Publish Date',
        'Operating Cash Flow',
        'Capital Expenditure',
        'Revenue',
        'Free Cash Flow',
        'Operating Cash Flow Growth',
        'Free Cash Flow Growth',
        'FCF Margin'
    ]
].copy()

print("Final Cash Flow feature shape:", cashflow_features.shape)

display(cashflow_features.head())

Final Cash Flow feature shape: (500, 11)


,SimFinId,Ticker,Fiscal Year,Publish Date,Operating Cash Flow,Capital Expenditure,Revenue,Free Cash Flow,Operating Cash Flow Growth,Free Cash Flow Growth,FCF Margin
0,18,GOOG,2020,2021-02-03,6.512400e+10,-2.228100e+10,1.825270e+11,4.284300e+10,NaN,NaN,0.234721
1,18,GOOG,2021,2022-02-02,9.165200e+10,-2.464000e+10,2.576370e+11,6.701200e+10,0.407346,0.564129,0.260102
2,18,GOOG,2022,2023-02-03,9.149500e+10,-3.148500e+10,2.828360e+11,6.001000e+10,-0.001713,-0.104489,0.212172
3,18,GOOG,2023,2024-01-31,1.017460e+11,-3.225100e+10,3.073940e+11,6.949500e+10,0.112039,0.158057,0.226078
4,18,GOOG,2024,2025-02-05,1.252990e+11,-5.253500e+10,3.500180e+11,7.276400e+10,0.231488,0.047039,0.207886


### Final Cash Flow Checks and Saving

A final structural check is performed to confirm that all company-year observations are preserved, company-year identifiers are unique, and Publish Date is available.

The completed Cash Flow feature dataset is then saved as a new CSV file without overwriting the original prepared financial dataset.

In [ ]:
print("Number of rows:", len(cashflow_features))
print("Unique companies:", cashflow_features['SimFinId'].nunique())

print(
    "Duplicate company-year records:",
    cashflow_features.duplicated(
        subset=['SimFinId', 'Fiscal Year']
    ).sum()
)

print(
    "Missing Publish Dates:",
    cashflow_features['Publish Date'].isnull().sum()
)

Number of rows: 500
Unique companies: 100
Duplicate company-year records: 0
Missing Publish Dates: 0


In [ ]:
cashflow_features_path = financial_path + "/cashflow_features.csv"

cashflow_features.to_csv(
    cashflow_features_path,
    index=False
)

print("Saved successfully:")
print(cashflow_features_path)

Saved successfully:
/content/drive/MyDrive/Data Handling and Machine Learning/week_03/outputs/financial_features/cashflow_features.csv


In [13]:
missing_fcf = cashflow_df[
    cashflow_df['Free Cash Flow'].isnull()
][
    ['Ticker', 'SimFinId', 'Fiscal Year',
     'Operating Cash Flow', 'Capital Expenditure', 'Free Cash Flow']
]

print("Missing FCF rows:", len(missing_fcf))
display(missing_fcf)

Missing FCF rows: 18


,Ticker,SimFinId,Fiscal Year,Operating Cash Flow,Capital Expenditure,Free Cash Flow
6,ABNB,1214552,2021,2.313000e+09,NaN,NaN
7,ABNB,1214552,2022,3.430000e+09,NaN,NaN
8,ABNB,1214552,2023,3.884000e+09,NaN,NaN
9,ABNB,1214552,2024,4.518000e+09,NaN,NaN
15,AC,919941,2020,-2.794830e+08,NaN,NaN
16,AC,919941,2021,2.381940e+08,NaN,NaN
17,AC,919941,2022,-6.304200e+07,NaN,NaN
18,AC,919941,2023,1.486650e+08,NaN,NaN
19,AC,919941,2024,4.308400e+07,NaN,NaN
160,F,249937,2020,2.426900e+10,NaN,NaN


In [14]:
print(
    "Unique companies with missing FCF:",
    missing_fcf['Ticker'].nunique()
)

print("\nCompanies:")
print(missing_fcf['Ticker'].unique())

Unique companies with missing FCF: 5

Companies:
['ABNB' 'AC' 'F' 'HASI' 'OAK']
